# Training Eshmun-Zero for Masked Language Modeling

This notebook demonstrates how to train an `EshmunForMaskedLM` model from scratch using the Hugging Face `Trainer` API.

**Pipeline:**
1. Build/load a tokenizer
2. Instantiate the config and model
3. Load and tokenize a dataset
4. Set up the MLM data collator
5. Train with `Trainer`
6. Run a quick fill-mask sanity check

## 1. Imports

In [1]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

from eshmun.models.zero.configuration import EshmunZeroConfig
from eshmun.models.zero.models import EshmunForMaskedLM

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [2]:
os.environ['WANDB_PROJECT'] = 'Eshmun'
os.environ['WANDB_LOG_MODEL'] = 'end'

## 2. Tokenizer

The default `vocab_size=25` in the config suggests a character/protein-style alphabet rather than BPE. Replace this with your own tokenizer (e.g. one trained on Phoenician characters). For the sake of a runnable example, we build a tiny character-level tokenizer here.

In [3]:
tokenizer_path = 'facebook/esm2_t6_8M_UR50D'
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)

## 3. Config & Model

Use a small config so the notebook runs quickly. Scale up `hidden_size`, `num_hidden_layers`, etc. for real training.

In [4]:
config = EshmunZeroConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_size=384,
    embedding_size=2048,
    hidden_act='silu',
    num_hidden_layers=8,
    intermediate_size=512,
    num_attention_heads=8,
    max_position_embeddings=514,
    local_window_size=12,
    alpha_init=0.0,
    pad_token_id=tokenizer.pad_token_id,
    is_decoder=False,
)

model = EshmunForMaskedLM(config)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params/1e6:.2f}M")

Trainable parameters: 15.39M


## 4. Dataset

Here we use `khairi/SwissProtSequences-SSL`.

In [5]:
raw_datasets = load_dataset("khairi/SwissProtSequences-SSL")
raw_datasets = raw_datasets.filter(lambda ex: len(ex["Sequence"].strip()) > 0) \
    .filter(lambda ex: len(ex['Sequence']) >= 20 and len(ex['Sequence']) <= 512)
print(raw_datasets)

DatasetDict({
    train: Dataset({
        features: ['Entry', 'Sequence'],
        num_rows: 465272
    })
})


In [6]:
MAX_LEN = 512

def tokenize_fn(examples):
    return tokenizer(
        examples["Sequence"],
        truncation=True,
        max_length=MAX_LEN,
        return_special_tokens_mask=True,
    )

tokenized = raw_datasets.map(
    tokenize_fn,
    batched=True,
    remove_columns=["Sequence"],
    desc="Tokenizing",
)
tokenized

DatasetDict({
    train: Dataset({
        features: ['Entry', 'input_ids', 'special_tokens_mask', 'attention_mask'],
        num_rows: 465272
    })
})

In [7]:
lm_datasets = tokenized['train'].train_test_split(shuffle=True, train_size=463272)

lm_datasets

DatasetDict({
    train: Dataset({
        features: ['Entry', 'input_ids', 'special_tokens_mask', 'attention_mask'],
        num_rows: 463272
    })
    test: Dataset({
        features: ['Entry', 'input_ids', 'special_tokens_mask', 'attention_mask'],
        num_rows: 2000
    })
})

## 5. Data Collator

`DataCollatorForLanguageModeling` handles the BERT-style 80/10/10 masking strategy and creates `labels` with `-100` at non-masked positions.

In [8]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.25,
)

## 6. Training

Adjust `num_train_epochs`, batch size, and learning rate to taste. The numbers below are minimal so the notebook runs quickly.

In [9]:
training_args = TrainingArguments(
    output_dir="/tmp/eshmun-zero-mlm",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=8,
    learning_rate=1e-3,
    weight_decay=0.01,
    warmup_steps=600,
    lr_scheduler_type="cosine",
    logging_steps=100,
    optim='adamw_torch',
    eval_steps=100,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=3,
    fp16=torch.cuda.is_available(),
    report_to="wandb",
    run_name="eshmun-zero-10M"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["test"],
    data_collator=data_collator,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[2026-05-01 22:29:38,980] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/home/khairi/miniconda3/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/home/khairi/miniconda3/compiler_compat/ld: error in /usr/lib/gcc/x86_64-pc-linux-gnu/15.2.1/../../../../lib/Scrt1.o(.sframe); no .sframe will be created
/home/khairi/miniconda3/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


In [10]:
train_result = trainer.train()
trainer.save_model()
tokenizer.save_pretrained(training_args.output_dir)
train_result.metrics

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/khairi/.netrc.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss
500,2.689375,2.643405
1000,2.632057,2.627817
1500,617.851312,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 7. Evaluation

In [ ]:
import math

eval_metrics = trainer.evaluate()
eval_metrics["perplexity"] = math.exp(eval_metrics["eval_loss"])
eval_metrics

## 8. Sanity check: fill a mask

Quick qualitative test that the model produces something other than uniform noise.

In [ ]:
model.eval()
# device = next(model.parameters()).device

# text = f"the quick brown fox {tokenizer.mask_token} over the lazy dog"
# inputs = tokenizer(text, return_tensors="pt").to(device)

# with torch.no_grad():
#     logits = model(**inputs).logits

# mask_idx = (inputs.input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
# top_ids = logits[0, mask_idx].topk(5, dim=-1).indices[0]
# print("Top-5 predictions:", tokenizer.convert_ids_to_tokens(top_ids))

## 9. (Optional) Inspect alpha gates

Since Eshmun-Zero exposes a learnable per-layer gate between local and global attention, it's useful to inspect what the model learned.